In [ ]:
# CPU preparation only: no GPU training or image transfer in this cell.
from google.colab import drive
import shutil, json, os
drive.mount('/content/drive')
print(json.dumps({'cpu_prep':True,'drive_free':shutil.disk_usage('/content/drive/MyDrive').free,'scratch_free':shutil.disk_usage('/content').free}))

In [ ]:
from pathlib import Path
import json, os, uuid
root=Path('/content/drive/MyDrive/UROP/official_images_v1')
p=root/'public_images.tar.bz2'
with p.open('rb') as f: header=f.read(4)
assert header[:3]==b'BZh'
temp=root/('io_probe_'+uuid.uuid4().hex+'.tmp')
with temp.open('wb') as f:
    f.write(b'UROP durable storage probe'); f.flush(); os.fsync(f.fileno())
committed=temp.with_suffix('.verified')
os.replace(temp,committed)
assert committed.read_bytes()==b'UROP durable storage probe'
print(json.dumps({'state':'cloud_storage_probe_passed','source_bytes':p.stat().st_size,'bzip_header':header.decode('ascii'),'fsync_rename_readback':True,'probe_file':str(committed),'image_extraction_started':False}))

In [ ]:
from pathlib import Path
import json, os, time, hashlib, importlib.util
root=Path('/content/drive/MyDrive/UROP/official_images_v1')
root.mkdir(parents=True,exist_ok=True)
src=Path('/content/drive/MyDrive/Copy of public_images.tar.bz2')
dst=root/'public_images.tar.bz2'
if not dst.exists():
    assert src.exists() and src.stat().st_size==113623359224
    src.rename(dst)
assert dst.stat().st_size==113623359224
(root/'cloud_storage_receipt.json').write_text(json.dumps({'account':'panvee62@gmail.com','available_bytes_lower_bound':4000000000000,'source':'Signed-in Drive UI showed5.39TB total with under200GB used; statvfs reports scratch instead','observed_date':'2026-10-23'}))
if not (root/'archive_complete.json').exists():
    (root/'archive_complete.json').write_text(json.dumps({'route':'Google Drive Make a copy; original author file unchanged','original_id':'1cjY6HsHaSZuLVHywIxD5xQqng33J5S2b','private_copy_id':'12YYAoTT2Mnb1eEoIRhuVJ3WmTvFoJXoi','bytes':dst.stat().st_size,'mtime_ns':dst.stat().st_mtime_ns,'sha256':None,'status':'Cloud copy size confirmed; full compressed hash and CRC pending extraction'}))
script=root/'prepare_official_images_drive_v1.py'
script.write_text("\"\"\"CPU-only, resumable staging of the author archive on an approved mounted Drive.\n\nRun THROUGH THE COLAB NOTEBOOK only after Drive access is approved. No OAuth,\ncredentials, GPUs, sharing changes or paid resources are created by this script.\nDownload and extraction are separate bounded operations. Shards preserve bytes.\n\"\"\"\nimport argparse,bz2,csv,hashlib,io,json,os,shutil,tarfile,time,uuid\nfrom datetime import datetime,timezone\nfrom pathlib import Path\nfrom PIL import Image\n\nSOURCE_ID='1cjY6HsHaSZuLVHywIxD5xQqng33J5S2b'\nTSVS=[('train','multimodal_train.tsv','1XsOkD3yhxgWu8URMes0S9LVSwuFJ4pT6','9894cb4aa2f901c6f29b61ac3d95ce9f7e78c43be8062ae5deb1c219c82497bb',564000),('validation','multimodal_validate.tsv','1Z99QrwpthioZQY2U6HElmnx8jazf7-Kv','c70140ec376e083e3780b5b488c96c8933678ca5800139bfb78c92e95df6946e',59342),('test','multimodal_test_public.tsv','1p9EewIKVcFbipVRLZNGYc2JbSC7A0SWv','bc92378e070ca7b1a7497053d1c34cdc3f550be6232a76791a2c722d6d9bbf94',59319)]\n\ndef now():return datetime.now(timezone.utc).isoformat()\ndef sha(path):\n h=hashlib.sha256()\n with open(path,'rb') as f:\n  for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)\n return h.hexdigest()\ndef record(path,value):\n # Immutable generation files: consumers trust only a completed parse + hashes.\n with Path(path).open('x') as f:json.dump(value,f,indent=2,allow_nan=False);f.flush();os.fsync(f.fileno())\ndef require_drive(root):\n if not root.resolve().is_relative_to(Path('/content/drive/MyDrive/UROP/official_images_v1')):raise ValueError('Dedicated authorized Drive staging path required')\n if not Path('/content/drive/MyDrive').is_dir():raise RuntimeError('Drive is not mounted; obtain user permission first')\n import torch\n if torch.cuda.is_available():raise RuntimeError('CPU preparation requires accelerator None; preserve GPU allowance')\n root.mkdir(parents=True,exist_ok=True)\n quota=json.loads((root/'cloud_storage_receipt.json').read_text())\n if quota.get('available_bytes_lower_bound',0)<300*1024**3:raise RuntimeError('Need verified remote Drive quota >=300GiB')\n # Drive FUSE statvfs can report notebook scratch space instead of account quota.\n if shutil.disk_usage('/content').free<8*1024**3:raise RuntimeError('Need8GiB scratch margin for bounded staging')\n # Prove the basic Drive file/rename route before a long download.\n test=root/('write_test_'+uuid.uuid4().hex);moved=test.with_suffix('.moved')\n test.write_bytes(b'UROP storage preflight');os.replace(test,moved)\n if moved.read_bytes()!=b'UROP storage preflight':raise IOError('Drive readback mismatch')\n moved.unlink()\ndef expected_rows(root,download=False):\n result={}\n for split,name,fileid,digest,count in TSVS:\n  p=root/name\n  if not p.exists():\n   if not download:raise FileNotFoundError(p)\n   import gdown\n   if not gdown.download(id=fileid,output=str(p),resume=True,use_cookies=False):raise RuntimeError('Official TSV download failed')\n  if sha(p)!=digest:raise ValueError('Official TSV checksum mismatch '+name)\n  with p.open() as f:rows=list(csv.DictReader(f,delimiter='\\t'))\n  if len(rows)!=count:raise ValueError('Official count mismatch')\n  for r in rows:\n   ident=r['id']\n   if ident in result or not ident.isalnum() or int(r['6_way_label']) not in range(6) or not r['clean_title'].strip():raise ValueError('Official ID/label/text invalid')\n   result[ident]={'id':ident,'split':split,'clean_title':r['clean_title'],'6_way_label':int(r['6_way_label'])}\n return result\n\ndef download(root):\n require_drive(root);expected_rows(root,download=True)\n archive=root/'public_images.tar.bz2';receipt=root/'archive_complete.json'\n if receipt.exists():\n  d=json.loads(receipt.read_text());assert archive.stat().st_size==d['bytes'] and sha(archive)==d['sha256'];print('Verified complete source already present');return\n import gdown\n # Partial bytes remain in this Drive directory across runtime loss. gdown's\n # range-resume still needs real-source verification before calling it proven.\n result=gdown.download(id=SOURCE_ID,output=str(archive),resume=True,use_cookies=False)\n if not result or archive.stat().st_size<90_000_000_000:raise RuntimeError('Source incomplete or unexpected small response; preserve partial evidence')\n with archive.open('rb') as f:\n  if f.read(3)!=b'BZh':raise ValueError('Expected bzip2, not HTML')\n record(receipt,{'utc':now(),'source_id':SOURCE_ID,'bytes':archive.stat().st_size,'sha256':sha(archive),'status':'transport complete; TAR coverage/CRC not yet verified'})\n\nclass DigestReader:\n def __init__(self,stream):self.stream=stream;self.digest=hashlib.sha256();self.count=0\n def read(self,size=-1):\n  value=self.stream.read(size);self.digest.update(value);self.count+=len(value);return value\n def readable(self):return True\n def seekable(self):return False\n\ndef extract(root,expected,shard_bytes=2*1024**3,max_shards=None,max_seconds=None):\n \"\"\"Resume by rereading durable compressed source and skipping committed IDs.\n\n Never redownloads the source. Abandoned shards are not trusted or silently used.\n Commit shard bytes first, then the corresponding receipt; crash before the\n receipt leaves an orphan for diagnosis. Shard receipts bind each image hash.\n \"\"\"\n shardroot=root/'shards';shardroot.mkdir(parents=True,exist_ok=True);seen={};failures=[]\n for receipt in sorted(shardroot.glob('*.receipt.json')):\n  r=json.loads(receipt.read_text());p=shardroot/r['archive']\n  if not p.is_file() or sha(p)!=r['sha256']:raise ValueError('Committed shard corrupt '+p.name)\n  for im in r['images']:\n   if im['id'] in seen:raise ValueError('Duplicate committed ID')\n   seen[im['id']]=im['image_sha256']\n writer=None;path=None;images=[];written=0;committed=0;archive_ids=set();start=time.monotonic()\n def commit():\n  nonlocal writer,path,images,written,committed\n  if writer is None:return\n  writer.close();writer=None\n  receipt={'utc':now(),'archive':path.name,'bytes':path.stat().st_size,'sha256':sha(path),'images':images}\n  record(path.with_suffix('.receipt.json'),receipt)\n  for im in images:seen[im['id']]=im['image_sha256']\n  images=[];written=0;committed+=1\n try:\n  with (root/'public_images.tar.bz2').open('rb') as original:\n   reader=DigestReader(original)\n   with bz2.BZ2File(reader,mode='rb') as decompressed,tarfile.open(fileobj=decompressed,mode='r|') as source:\n    for member in source:\n     if max_seconds is not None and time.monotonic()-start>=max_seconds:break\n     if not member.isfile():continue\n     ident=Path(member.name).stem\n     if ident not in expected:continue\n     if ident in archive_ids:raise ValueError('Duplicate required source ID '+ident)\n     archive_ids.add(ident)\n     if ident in seen:continue\n     if member.size>512*1024**2:raise ValueError('Unexpected required image >512MiB '+ident)\n     stream=source.extractfile(member)\n     if stream is None:raise ValueError('Unreadable required member '+ident)\n     raw=stream.read();digest=hashlib.sha256(raw).hexdigest()\n     try:\n      with Image.open(io.BytesIO(raw)) as im:im.load();im.convert('RGB').load()\n     except Exception as e:failures.append({'id':ident,'error':str(e)});continue\n     if writer is not None and written+len(raw)>shard_bytes:\n      commit()\n      if max_shards is not None and committed>=max_shards:break\n     if writer is None:\n      path=shardroot/('images_'+uuid.uuid4().hex+'.tar');writer=tarfile.open(path,'w')\n     suffix=Path(member.name).suffix.lower();name='images/'+ident+suffix\n     info=tarfile.TarInfo(name);info.size=len(raw);info.mtime=0;writer.addfile(info,io.BytesIO(raw))\n     images.append({**expected[ident],'member':name,'image_sha256':digest,'bytes':len(raw),'source_member':member.name});written+=len(raw)\n     if (len(seen)+len(images))%1000==0:\n      status={'utc':now(),'state':'extracting','usable_committed':len(seen),'usable_in_open_shard':len(images),'required':len(expected),'elapsed_seconds':time.monotonic()-start}\n      print(json.dumps(status),flush=True)\n      if str(root).startswith('/content/drive/') and shutil.disk_usage('/content').free<8*1024**3:raise RuntimeError('Scratch margin fell below8GiB; preserve committed shards')\n    else:\n     commit()\n     while decompressed.read(1024*1024):pass # consume bzip2 footer and verify its CRC\n     if reader.count!=(root/'public_images.tar.bz2').stat().st_size:raise ValueError('Compressed source was not completely consumed')\n     missing=sorted(set(expected)-set(seen))\n     report={'utc':now(),'image_gate_passed':not missing and not failures,'required':len(expected),'usable':len(seen),'missing_ids':missing,'decode_failures':failures,'compressed_archive_sha256':reader.digest.hexdigest(),'compressed_archive_bytes':reader.count,'elapsed_seconds':time.monotonic()-start,'status':'Extraction complete; private Kaggle import/runtime hash audit still required'}\n     record(root/('extraction_'+uuid.uuid4().hex+'.json'),report);return report\n  commit();return {'status':'bounded extraction checkpoint','committed_shards':committed,'usable_committed':len(seen),'image_gate_passed':False}\n finally:\n  if writer is not None:writer.close() # orphan without receipt remains untrusted\n\ndef main():\n ap=argparse.ArgumentParser();ap.add_argument('--mode',choices=['download','extract'],required=True);ap.add_argument('--root',type=Path,default=Path('/content/drive/MyDrive/UROP/official_images_v1'));ap.add_argument('--max-shards',type=int,default=128);ap.add_argument('--max-seconds',type=int,default=14400);a=ap.parse_args();require_drive(a.root)\n if a.mode=='download':download(a.root)\n else:\n  archive=json.loads((a.root/'archive_complete.json').read_text());p=a.root/'public_images.tar.bz2'\n  if p.stat().st_size!=archive['bytes']:raise ValueError('Source archive size changed')\n  if archive.get('sha256') and sha(p)!=archive['sha256']:raise ValueError('Source archive integrity failed')\n  if archive.get('mtime_ns') and p.stat().st_mtime_ns!=archive['mtime_ns']:raise ValueError('Cloud copy modified since staging')\n  print(json.dumps(extract(a.root,expected_rows(a.root),max_shards=a.max_shards,max_seconds=a.max_seconds),indent=2))\nif __name__=='__main__':main()\n")
print(json.dumps({'state':'cloud_source_staged','source_bytes':dst.stat().st_size,'script_sha256':hashlib.sha256(script.read_bytes()).hexdigest(),'gdown_available':importlib.util.find_spec('gdown') is not None,'image_extraction_started':False}))